In [ ]:
tickers = [
  {
    "ticker": "MDLZ",
    "position": "long",
    "thesis": "Mondelez positioned for sustained growth through pricing power and strategic acquisitions in emerging markets, with robust margin expansion offsetting commodity pressures.",
    "key_drivers": "Q3 2024 organic revenue +5.4%, EPS +28.6%, Evirth acquisition enhances China exposure; strong cash flow $2.5B YTD supports reinvestment.",
    "key_risks": "Cocoa inflation lag (Memory 2), potential volume softness in indulgent categories from GLP-1 trends (Memory 10); high retailer concentration (Memory 5).",
    "valuation_snapshot": "Current PE 18x (above industry avg 5.37, but justified by growth); EV/EBITDA 14.3x; undervalued relative to forward EPS CAGR 0.61.",
    "conviction": 0.88
  },
  {
    "ticker": "K",
    "position": "long",
    "thesis": "Kellanova's snacking focus drives outperformance with strong Q3 momentum, leveraging brand strength in a resilient category amid portfolio optimization.",
    "key_drivers": "Q3 sales +5%, adjusted operating profit +16%, EPS +18%; snacking leadership with Pringles/Cheez-It provides defensive growth.",
    "key_risks": "Merger-related costs from Mars deal; commodity exposure in snacks (Memory 2); trade spend opacity (Memory 6).",
    "valuation_snapshot": "Forward PE ~15x vs industry 12.66; book_to_market 0.45 (at avg); attractive FCF yield 3.5%.",
    "conviction": 0.85
  },
  {
    "ticker": "CPB",
    "position": "long",
    "thesis": "Campbell's margin recovery and snacks portfolio integration position it for profitability inflection, with cost savings offsetting sales softness.",
    "key_drivers": "Q3 margins +40bps to 14.9%, approaching 15% goal; $900M YTD cash flow; snacks target 17% margins via Hostess/SOS integration.",
    "key_risks": "Pop Secret divestiture EPS dilution; weak consumer trends (Memory 8); leverage 3.7x post-acq.",
    "valuation_snapshot": "Trailing PE 11x (in line with avg); EV/EBITDA 11.5x; normalized for acquisition impacts.",
    "conviction": 0.82
  },
  {
    "ticker": "POST",
    "position": "long",
    "thesis": "Post Holdings benefits from packaged foods resilience and growth in high-margin categories, supported by stable execution.",
    "key_drivers": "Strong Q3 packaged growth; quality margins above industry; low commodity exposure relative to peers.",
    "key_risks": "Private label pressure in cereals (Memory 8); moderate liquidity increases squeeze potential (Task 6).",
    "valuation_snapshot": "PE 17x (slight premium to avg); ROIC z-score +0.5; undervalued on forward growth.",
    "conviction": 0.77
  },
  {
    "ticker": "GIS",
    "position": "long",
    "thesis": "General Mills' cost discipline and HMM initiatives drive margin restoration to pre-pandemic levels, providing stability in a challenged environment.",
    "key_drivers": "Q3 adjusted EPS $1.17 (+YoY), gross margin +20bps to 34%; ongoing productivity savings.",
    "key_risks": "Weak sales volumes; GLP-1 impacts on indulgent products (Memory 10); scanner vs shipment gaps (Memory 15).",
    "valuation_snapshot": "Trailing PE 11x (at avg); dividend yield 3.7%; EV/EBITDA 14x in line.",
    "conviction": 0.74
  },
  {
    "ticker": "HSY",
    "position": "short",
    "thesis": "Hershey faces near-term headwinds from volume declines and margin compression due to cocoa costs and consumer shifts, eroding profitability.",
    "key_drivers": "Q3 sales/margins below expectations; negative product mix and deleverage from lower volumes.",
    "key_risks": "High cocoa exposure (Memory 2); indulgent category sensitivity to GLP-1 (Memory 10); retailer concentration (Memory 5).",
    "valuation_snapshot": "Forward PE 20x (premium to avg); EV/EBITDA 16x; overvalued given weak momentum.",
    "conviction": 0.42
  },
  {
    "ticker": "KHC",
    "position": "short",
    "thesis": "Kraft Heinz contends with leverage and commodity pressures in snacks/beverages, limiting upside amid limited visibility on Q3 performance.",
    "key_drivers": "Elevated debt (~4x leverage); ongoing input cost inflation in key categories (Memory 2).",
    "key_risks": "Regulatory scrutiny on additives (Memory 9); trade spend risks (Memory 6); potential private label gains (Memory 8).",
    "valuation_snapshot": "Trailing PE 12x (slight discount); book_to_market 0.76 (high value but low growth); EV/EBITDA 10x.",
    "conviction": 0.40
  },
  {
    "ticker": "CAG",
    "position": "short",
    "thesis": "Conagra's supply chain volatility and private label competition pressure margins, with mixed Q4 results signaling ongoing challenges.",
    "key_drivers": "Q4 EPS beat but supply chain issues persist; PL erosion in frozen veggies (Memory 8).",
    "key_risks": "High trade spend (Memory 6); channel mix shifts (Memory 12); inventory accounting distortions (Memory 14).",
    "valuation_snapshot": "PE 10x (below avg, but declining ROIC); EV/EBITDA 12x; attractive on value but risks weigh.",
    "conviction": 0.39
  },
  {
    "ticker": "FDP",
    "position": "short",
    "thesis": "Fresh Del Monte's weak growth and exposure to commodity cycles position it for continued underperformance in a competitive landscape.",
    "key_drivers": "Below-benchmark revenue growth; medium liquidity but high event risk from perishables.",
    "key_risks": "Avian flu/protein cycles (Memory 13); low visibility on catalysts; private label in produce (Memory 8).",
    "valuation_snapshot": "Trailing PE 9x (undervalued); but low ROIC z-score -0.3; high vol 0.5.",
    "conviction": 0.36
  }
]

In [50]:
import sys
sys.path.append('..')
from app.db.core.models.market_data_models import *
from app.db.core.db_config import MarketSession
from app.repositories.price_data import fetch_bulk_price_data_for_tickers


In [51]:
from datetime import datetime, timedelta
from app.core.calculations.returns.calculator import ReturnsCalculator
import pandas as pd

# Extract ticker symbols and positions
ticker_symbols = [t['ticker'] for t in tickers]
ticker_positions = {t['ticker']: t['position'] for t in tickers}

# Calculate dates for 365 days lookback
end_date = datetime.now()
start_date = end_date - timedelta(days=365)

# Format dates for the fetch function
start_date_str = start_date.strftime('%Y-%m-%d')
end_date_str = end_date.strftime('%Y-%m-%d')

# Fetch bulk price data
price_data_map = fetch_bulk_price_data_for_tickers(
    tickers=ticker_symbols,
    start_date_str=start_date_str,
    end_date_str=end_date_str,
    frequency='daily'
)

# Calculate returns for each ticker
returns_data = []
for ticker in ticker_symbols:
    position = ticker_positions[ticker]
    
    if ticker in price_data_map:
        close_prices = price_data_map[ticker]
        if not close_prices.empty:
            # Calculate total return over period
            daily_returns = ReturnsCalculator.daily_price_returns(close_prices)
            raw_return = (1 + daily_returns).prod() - 1
            
            # Position return (long/short adjusted)
            position_return = raw_return if position == 'long' else -raw_return
            
            # Capped position return (cap at -5%)
            capped_return = max(position_return, -0.075)

            returns_data.append({
                'ticker': ticker,
                'position': position,
                'raw_return': raw_return,
                'position_return': position_return,
                'capped_return': capped_return
            })

# Create DataFrame
returns_df = pd.DataFrame(returns_data)
returns_df['raw_return_pct'] = returns_df['raw_return'] * 100
returns_df['position_return_pct'] = returns_df['position_return'] * 100
returns_df['capped_return_pct'] = returns_df['capped_return'] * 100

# Display individual ticker results
print(f"\nReturns Analysis ({start_date_str} to {end_date_str})")
print("=" * 100)
print(returns_df[['ticker', 'position', 'raw_return_pct', 'position_return_pct', 'capped_return_pct']].to_string(index=False))

# Calculate overall portfolio returns
overall_return = (returns_df['position_return'] + 1).prod() - 1
overall_return_capped = (returns_df['capped_return'] + 1).prod() - 1

print("\n" + "=" * 100)
print(f"\nOverall Portfolio Return (Compounded): {overall_return*100:.2f}%")
print(f"Overall Portfolio Return (Capped at -5%): {overall_return_capped*100:.2f}%")

returns_df


Returns Analysis (2024-10-02 to 2025-10-02)
ticker position  raw_return_pct  position_return_pct  capped_return_pct
  MDLZ     long      -12.484446           -12.484446          -7.500000
     K     long        1.551061             1.551061           1.551061
   CPB     long      -34.200627           -34.200627          -7.500000
  POST     long       -8.471038            -8.471038          -7.500000
   GIS     long      -31.869031           -31.869031          -7.500000
   HSY    short       -4.420661             4.420661           4.420661
   KHC    short      -24.922316            24.922316          24.922316
   CAG    short      -39.826551            39.826551          39.826551
   FDP    short       17.846154           -17.846154          -7.500000


Overall Portfolio Return (Compounded): -45.36%
Overall Portfolio Return (Capped at -5%): 25.43%


,ticker,position,raw_return,position_return,capped_return,raw_return_pct,position_return_pct,capped_return_pct
0,MDLZ,long,-0.124844,-0.124844,-0.075000,-12.484446,-12.484446,-7.500000
1,K,long,0.015511,0.015511,0.015511,1.551061,1.551061,1.551061
2,CPB,long,-0.342006,-0.342006,-0.075000,-34.200627,-34.200627,-7.500000
3,POST,long,-0.084710,-0.084710,-0.075000,-8.471038,-8.471038,-7.500000
4,GIS,long,-0.318690,-0.318690,-0.075000,-31.869031,-31.869031,-7.500000
5,HSY,short,-0.044207,0.044207,0.044207,-4.420661,4.420661,4.420661
6,KHC,short,-0.249223,0.249223,0.249223,-24.922316,24.922316,24.922316
7,CAG,short,-0.398266,0.398266,0.398266,-39.826551,39.826551,39.826551
8,FDP,short,0.178462,-0.178462,-0.075000,17.846154,-17.846154,-7.500000
